In [1]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("GPU not detected!")


PyTorch: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


In [3]:
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes scipy matplotlib pandas

In [4]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb
import accelerate

print("========== ENVIRONMENT ==========")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\n========== LIBRARY VERSIONS ==========")
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bnb.__version__)
print("Accelerate:", accelerate.__version__)

print("\nAll libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


========== ENVIRONMENT ==========
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4

========== LIBRARY VERSIONS ==========
Transformers: 5.24.1
Datasets: 5.2.0
PEFT: 0.22.0
TRL: 1.19.0
BitsAndBytes: 0.50.2
Accelerate: 1.16.0

All libraries imported successfully!


In [5]:
%pip install -q -U bottleneck

In [6]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split

BASE_DIR = "/content/Professional_Response_Generator"

DATA_DIR = os.path.join(BASE_DIR, "dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project directories created!")
print("Dataset:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

Project directories created!
Dataset: /content/Professional_Response_Generator/dataset
Outputs: /content/Professional_Response_Generator/outputs


In [7]:
additional_examples = [

    # Professional emails
    {
        "category": "professional_email",
        "user": "i want to apply for the software developer position in your company",
        "assistant": "I am writing to express my interest in the Software Developer position at your company. I would appreciate the opportunity to be considered for this role."
    },
    {
        "category": "professional_email",
        "user": "please find attached my internship completion certificate",
        "assistant": "Please find attached my internship completion certificate for your reference. Kindly let me know if any additional documentation is required."
    },
    {
        "category": "professional_email",
        "user": "i am following up on my previous email",
        "assistant": "I am writing to follow up on my previous email. I would appreciate an update whenever convenient."
    },
    {
        "category": "professional_email",
        "user": "can you please tell me when the results will be announced",
        "assistant": "Could you please let me know when the results are expected to be announced?"
    },
    {
        "category": "professional_email",
        "user": "thanks for giving me this opportunity",
        "assistant": "Thank you for providing me with this opportunity. I sincerely appreciate your consideration."
    },

    # Professional messages
    {
        "category": "professional_message",
        "user": "i will reach office 15 minutes late",
        "assistant": "I wanted to inform you that I may arrive at the office approximately 15 minutes late. I apologize for any inconvenience."
    },
    {
        "category": "professional_message",
        "user": "please let me know when you are free",
        "assistant": "Please let me know when you are available so that we can connect."
    },
    {
        "category": "professional_message",
        "user": "i have sent you the file on mail",
        "assistant": "I have shared the file with you via email. Kindly check your inbox when convenient."
    },
    {
        "category": "professional_message",
        "user": "sorry i missed your call",
        "assistant": "Apologies for missing your call. Please let me know a convenient time to connect."
    },
    {
        "category": "professional_message",
        "user": "i will update you once the work is completed",
        "assistant": "I will share an update with you as soon as the work is completed."
    },

    # Academic communication
    {
        "category": "academic_communication",
        "user": "sir our project presentation is ready can we show it to you",
        "assistant": "Sir, our project presentation is ready. Would it be possible to schedule a time to present it to you and receive your feedback?"
    },
    {
        "category": "academic_communication",
        "user": "i have uploaded the assignment please verify it",
        "assistant": "I have uploaded the assignment for submission. Could you please verify it and let me know if any corrections are needed?"
    },
    {
        "category": "academic_communication",
        "user": "sir i need clarification about question number 4",
        "assistant": "Sir, I would appreciate some clarification regarding Question 4. Could you please explain the expected approach?"
    },
    {
        "category": "academic_communication",
        "user": "our team needs an extension because the project is not finished",
        "assistant": "Our team would like to request an extension for the project submission, as some of the work is still in progress. We would appreciate your consideration."
    },
    {
        "category": "academic_communication",
        "user": "sir please sign the project report so we can submit it",
        "assistant": "Sir, could you please review and sign the project report so that we can proceed with the submission?"
    },

    # Corporate communication
    {
        "category": "corporate_communication",
        "user": "i am available for the interview next week",
        "assistant": "I am available to attend the interview next week. Please let me know your preferred date and time."
    },
    {
        "category": "corporate_communication",
        "user": "i completed the development task and pushed the code",
        "assistant": "I have completed the development task and pushed the changes to the repository. Please let me know if any further modifications are required."
    },
    {
        "category": "corporate_communication",
        "user": "can we postpone the meeting to tomorrow",
        "assistant": "Would it be possible to reschedule the meeting to tomorrow? Please let me know if that works for you."
    },
    {
        "category": "corporate_communication",
        "user": "i want to know the status of my job application",
        "assistant": "I am writing to inquire about the current status of my job application. I would appreciate any updates you can share."
    },
    {
        "category": "corporate_communication",
        "user": "i have some suggestions to improve the current process",
        "assistant": "I have a few suggestions that may help improve the current process. I would be happy to discuss them with the team."
    },

    # Tone correction
    {
        "category": "tone_correction",
        "user": "you didn't reply to my message",
        "assistant": "I wanted to follow up on my previous message, as I have not yet received a response. Please let me know when you have a chance."
    },
    {
        "category": "tone_correction",
        "user": "this work is wrong do it again",
        "assistant": "There appear to be some issues with the current work. Could you please review it and make the necessary corrections?"
    },
    {
        "category": "tone_correction",
        "user": "i don't understand what you are saying",
        "assistant": "I would appreciate it if you could clarify your explanation, as I am having some difficulty understanding the details."
    },
    {
        "category": "tone_correction",
        "user": "send the document quickly",
        "assistant": "Could you please share the document at your earliest convenience?"
    },
    {
        "category": "tone_correction",
        "user": "you gave me the wrong information",
        "assistant": "I believe there may be an error in the information shared. Could you please review it and confirm the correct details?"
    }
]

print("Additional examples prepared:", len(additional_examples))

Additional examples prepared: 25


In [10]:

import json
import os

SYSTEM_PROMPT = (
    "You are a professional communication assistant. "
    "Rewrite informal or rough messages into clear, polished, "
    "natural professional communication. Preserve the original "
    "meaning, do not invent facts, and use an appropriate tone."
)

examples = [
    ("sir please check the report and tell me changes",
     "Sir, could you please review the report and let me know if any changes are required?"),

    ("i cannot attend the meeting today because of some work",
     "I will be unable to attend today's meeting due to a prior commitment. I apologize for the inconvenience."),

    ("send me the details of the internship",
     "Could you please share the details regarding the internship opportunity?"),

    ("i have completed the task please check",
     "I have completed the assigned task. Could you please review it and share your feedback?"),

    ("sir can we meet tomorrow to discuss the project",
     "Sir, would it be possible to schedule a meeting tomorrow to discuss the project?"),

    ("i am interested in this job please consider my application",
     "I am interested in the position and would appreciate your consideration of my application."),

    ("sorry for late reply i was busy",
     "Apologies for the delayed response. I was occupied with other commitments. Thank you for your patience."),

    ("please approve my leave for two days",
     "I would like to request approval for two days of leave. Please let me know if any additional information is required."),

    ("we finished the project and uploaded everything",
     "We have completed the project and uploaded all the required files."),

    ("can you send the meeting link",
     "Could you please share the meeting link when convenient?"),

    ("i need more time to complete the assignment",
     "I would like to request an extension to complete the assignment. I would appreciate your consideration."),

    ("thank you for selecting me for the interview",
     "Thank you for considering my application and providing me with the opportunity to interview for the position."),

    ("i have attached my resume please check it",
     "I have attached my resume for your review. Please let me know if you require any additional information."),

    ("we need to discuss some problems in the project",
     "We would like to discuss a few issues related to the project and identify the necessary next steps."),

    ("please confirm if you received my document",
     "Could you please confirm whether you have received the document?")
]

dataset = []

for user_text, assistant_text in examples:
    dataset.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    })

print("Starter dataset recreated!")
print("Total examples:", len(dataset))


Starter dataset recreated!
Total examples: 15


In [14]:
import os
import json

# ==========================================
# 1. PROJECT SETUP
# ==========================================

BASE_DIR = "/content/Professional_Response_Generator"
DATA_DIR = os.path.join(BASE_DIR, "dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a professional communication assistant. "
    "Rewrite informal or rough messages into clear, polished, "
    "natural professional communication. Preserve the original "
    "meaning, do not invent facts, and use an appropriate tone."
)

# ==========================================
# 2. ALL 40 TRAINING EXAMPLES
# ==========================================

examples = [

    # Professional emails
    ("professional_email", "i am interested in this job please consider my application",
     "I am interested in the position and would appreciate your consideration of my application."),

    ("professional_email", "send me the details of the internship",
     "Could you please share the details regarding the internship opportunity?"),

    ("professional_email", "i am following up on my previous email",
     "I am writing to follow up on my previous email. I would appreciate an update whenever convenient."),

    ("professional_email", "thanks for giving me this opportunity",
     "Thank you for providing me with this opportunity. I sincerely appreciate your consideration."),

    ("professional_email", "i have attached my resume please check it",
     "I have attached my resume for your review. Please let me know if you require any additional information."),

    ("professional_email", "please find attached my internship completion certificate",
     "Please find attached my internship completion certificate for your reference. Kindly let me know if any additional documentation is required."),

    ("professional_email", "can you please tell me when the results will be announced",
     "Could you please let me know when the results are expected to be announced?"),

    ("professional_email", "i want to apply for the software developer position in your company",
     "I am writing to express my interest in the Software Developer position at your company. I would appreciate the opportunity to be considered for this role."),

    ("professional_email", "please confirm if you received my document",
     "Could you please confirm whether you have received the document?"),

    # Professional messages
    ("professional_message", "sorry for late reply i was busy",
     "Apologies for the delayed response. I was occupied with other commitments. Thank you for your patience."),

    ("professional_message", "can you send the meeting link",
     "Could you please share the meeting link when convenient?"),

    ("professional_message", "please let me know when you are free",
     "Please let me know when you are available so that we can connect."),

    ("professional_message", "i will reach office 15 minutes late",
     "I wanted to inform you that I may arrive at the office approximately 15 minutes late. I apologize for any inconvenience."),

    ("professional_message", "i have sent you the file on mail",
     "I have shared the file with you via email. Kindly check your inbox when convenient."),

    ("professional_message", "sorry i missed your call",
     "Apologies for missing your call. Please let me know a convenient time to connect."),

    ("professional_message", "i will update you once the work is completed",
     "I will share an update with you as soon as the work is completed."),

    ("professional_message", "i cannot attend the meeting today because of some work",
     "I will be unable to attend today's meeting due to a prior commitment. I apologize for the inconvenience."),

    # Academic communication
    ("academic_communication", "sir please check the report and tell me changes",
     "Sir, could you please review the report and let me know if any changes are required?"),

    ("academic_communication", "sir can we meet tomorrow to discuss the project",
     "Sir, would it be possible to schedule a meeting tomorrow to discuss the project?"),

    ("academic_communication", "i need more time to complete the assignment",
     "I would like to request an extension to complete the assignment. I would appreciate your consideration."),

    ("academic_communication", "we need to discuss some problems in the project",
     "We would like to discuss a few issues related to the project and identify the necessary next steps."),

    ("academic_communication", "sir our project presentation is ready can we show it to you",
     "Sir, our project presentation is ready. Would it be possible to schedule a time to present it to you and receive your feedback?"),

    ("academic_communication", "i have uploaded the assignment please verify it",
     "I have uploaded the assignment for submission. Could you please verify it and let me know if any corrections are needed?"),

    ("academic_communication", "sir i need clarification about question number 4",
     "Sir, I would appreciate some clarification regarding Question 4. Could you please explain the expected approach?"),

    ("academic_communication", "sir please sign the project report so we can submit it",
     "Sir, could you please review and sign the project report so that we can proceed with the submission?"),

    ("academic_communication", "our team needs an extension because the project is not finished",
     "Our team would like to request an extension for the project submission, as some of the work is still in progress. We would appreciate your consideration."),

    # Corporate communication
    ("corporate_communication", "i have completed the task please check",
     "I have completed the assigned task. Could you please review it and share your feedback?"),

    ("corporate_communication", "we finished the project and uploaded everything",
     "We have completed the project and uploaded all the required files."),

    ("corporate_communication", "i am available for the interview next week",
     "I am available to attend the interview next week. Please let me know your preferred date and time."),

    ("corporate_communication", "i completed the development task and pushed the code",
     "I have completed the development task and pushed the changes to the repository. Please let me know if any further modifications are required."),

    ("corporate_communication", "can we postpone the meeting to tomorrow",
     "Would it be possible to reschedule the meeting to tomorrow? Please let me know if that works for you."),

    ("corporate_communication", "i want to know the status of my job application",
     "I am writing to inquire about the current status of my job application. I would appreciate any updates you can share."),

    ("corporate_communication", "i have some suggestions to improve the current process",
     "I have a few suggestions that may help improve the current process. I would be happy to discuss them with the team."),

    # Tone correction
    ("tone_correction", "you didn't reply to my message",
     "I wanted to follow up on my previous message, as I have not yet received a response. Please let me know when you have a chance."),

    ("tone_correction", "this work is wrong do it again",
     "There appear to be some issues with the current work. Could you please review it and make the necessary corrections?"),

    ("tone_correction", "i don't understand what you are saying",
     "I would appreciate it if you could clarify your explanation, as I am having some difficulty understanding the details."),

    ("tone_correction", "send the document quickly",
     "Could you please share the document at your earliest convenience?"),

    ("tone_correction", "you gave me the wrong information",
     "I believe there may be an error in the information shared. Could you please review it and confirm the correct details?"),

    ("tone_correction", "please approve my leave for two days",
     "I would like to request approval for two days of leave. Please let me know if any additional information is required.")
]

# ==========================================
# 3. CREATE CHAT FORMAT DATASET
# ==========================================

combined = []

for category, user_text, assistant_text in examples:

    record = {
        "category": category,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            },
            {
                "role": "assistant",
                "content": assistant_text
            }
        ]
    }

    combined.append(record)

# ==========================================
# 4. REMOVE DUPLICATES
# ==========================================

unique = {}

for item in combined:

    user_text = (
        item["messages"][1]["content"].strip().lower()
    )

    unique[user_text] = item

combined = list(unique.values())

# ==========================================
# 5. VALIDATE DATASET
# ==========================================

for item in combined:

    assert len(item["messages"]) == 3

    assert item["messages"][0]["role"] == "system"
    assert item["messages"][1]["role"] == "user"
    assert item["messages"][2]["role"] == "assistant"

    assert all(
        message["content"].strip()
        for message in item["messages"]
    )

# ==========================================
# 6. SAVE JSONL FILE
# ==========================================

jsonl_path = os.path.join(
    DATA_DIR,
    "professional_responses.jsonl"
)

with open(jsonl_path, "w", encoding="utf-8") as f:

    for item in combined:

        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

# ==========================================
# 7. READ AND VERIFY
# ==========================================

with open(jsonl_path, "r", encoding="utf-8") as f:

    saved_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(saved_records) == len(combined)

# ==========================================
# 8. DISPLAY RESULTS
# ==========================================

print("=" * 45)
print("DATASET CREATED SUCCESSFULLY")
print("=" * 45)

print("Total examples:", len(saved_records))
print("Unique examples:", len(unique))
print("Duplicates removed:", len(examples) - len(combined))

print("\nCATEGORY DISTRIBUTION:")

for category in sorted(set(x["category"] for x in combined)):
    count = sum(
        1 for x in combined if x["category"] == category
    )
    print(category, ":", count)

print("\nFILE LOCATION:")
print(jsonl_path)

print("\nFIRST TRAINING EXAMPLE:")
print(json.dumps(saved_records[0], indent=2, ensure_ascii=False))

print("\nDATASET VALIDATION: PASSED")

DATASET CREATED SUCCESSFULLY
Total examples: 39
Unique examples: 39
Duplicates removed: 0

CATEGORY DISTRIBUTION:
academic_communication : 9
corporate_communication : 7
professional_email : 9
professional_message : 8
tone_correction : 6

FILE LOCATION:
/content/Professional_Response_Generator/dataset/professional_responses.jsonl

FIRST TRAINING EXAMPLE:
{
  "category": "professional_email",
  "messages": [
    {
      "role": "system",
      "content": "You are a professional communication assistant. Rewrite informal or rough messages into clear, polished, natural professional communication. Preserve the original meaning, do not invent facts, and use an appropriate tone."
    },
    {
      "role": "user",
      "content": "i am interested in this job please consider my application"
    },
    {
      "role": "assistant",
      "content": "I am interested in the position and would appreciate your consideration of my application."
    }
  ]
}

DATASET VALIDATION: PASSED


In [15]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

backup_dir = "/content/drive/MyDrive/Professional_Response_Generator"
os.makedirs(backup_dir, exist_ok=True)

shutil.copy2(
    jsonl_path,
    os.path.join(backup_dir, "professional_responses_backup.jsonl")
)

print("Dataset backup completed!")

Mounted at /content/drive
Dataset backup completed!


In [16]:
# ==========================================
# STEP 4 — EXPAND DATASET
# ==========================================

import json
import os

# Load our existing validated dataset
jsonl_path = "/content/Professional_Response_Generator/dataset/professional_responses.jsonl"

with open(jsonl_path, "r", encoding="utf-8") as f:
    existing_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Existing examples:", len(existing_records))


# ------------------------------------------
# Additional curated examples
# ------------------------------------------

new_examples = [

    # ======================================
    # PROFESSIONAL EMAILS
    # ======================================

    ("professional_email",
     "i am writing to ask about the status of my interview",
     "I am writing to inquire about the status of my interview. I would appreciate any updates you may be able to share."),

    ("professional_email",
     "please let me know if there are any openings in your company",
     "I would appreciate it if you could let me know about any suitable openings currently available within your organization."),

    ("professional_email",
     "i would like to request information about the position",
     "I would like to request additional information regarding the position. I would appreciate any details you can share."),

    ("professional_email",
     "thank you for your response",
     "Thank you for your response. I appreciate you taking the time to address my query."),

    ("professional_email",
     "i am sending the documents you requested",
     "Please find attached the documents you requested. Kindly let me know if any additional information is required."),

    ("professional_email",
     "please let me know if you need anything else from me",
     "Please let me know if you require any additional information or documentation from my side."),

    ("professional_email",
     "i wanted to ask if the position is still available",
     "I wanted to inquire whether the position is still available. I would appreciate any information you can provide."),

    ("professional_email",
     "can you please provide feedback on my application",
     "Could you please share any feedback regarding my application? I would greatly appreciate your input."),

    ("professional_email",
     "i am available whenever you want to schedule the interview",
     "I am available to attend the interview at your convenience. Please let me know the preferred date and time."),

    ("professional_email",
     "i apologize for not submitting the document earlier",
     "I apologize for the delay in submitting the document. Thank you for your understanding."),

    ("professional_email",
     "please let me know the next steps",
     "Could you please let me know the next steps in the process?"),

    ("professional_email",
     "i would like to schedule a meeting regarding the project",
     "I would like to schedule a meeting to discuss the project. Please let me know a convenient time for you."),

    ("professional_email",
     "thank you for considering my application",
     "Thank you for considering my application. I sincerely appreciate the opportunity."),

    ("professional_email",
     "i am interested in discussing this opportunity further",
     "I would be interested in discussing this opportunity further. Please let me know a suitable time to connect."),

    ("professional_email",
     "please confirm the date and time of the meeting",
     "Could you please confirm the date and time scheduled for the meeting?"),

    ("professional_email",
     "i have completed the requested changes",
     "I have completed the requested changes. Kindly review the updated version and let me know if anything further is required."),

    ("professional_email",
     "i would like to know more about the internship responsibilities",
     "I would appreciate more information regarding the responsibilities and expectations associated with the internship."),

    ("professional_email",
     "please accept my apologies for the delay",
     "Please accept my apologies for the delay. I appreciate your patience and understanding."),

    ("professional_email",
     "i am sharing the updated version of the report",
     "I am sharing the updated version of the report for your review. Please let me know if any further changes are required."),

    ("professional_email",
     "could you please clarify the deadline",
     "Could you please clarify the submission deadline so that I can plan accordingly?"),

    ("professional_email",
     "i would like to thank you for your guidance",
     "I would like to sincerely thank you for your guidance and support."),

    ("professional_email",
     "please let me know if the meeting can be conducted online",
     "Could you please let me know whether the meeting can be conducted online?"),

    ("professional_email",
     "i am unable to attend the scheduled meeting",
     "I regret that I will be unable to attend the scheduled meeting. I apologize for any inconvenience caused."),

    ("professional_email",
     "can you share the required documents list",
     "Could you please share the list of documents required for the process?"),

    ("professional_email",
     "i am writing regarding the opportunity discussed earlier",
     "I am writing regarding the opportunity we discussed earlier and would appreciate any further information you can provide."),

    ("professional_email",
     "please review the attached proposal",
     "Please find the proposal attached for your review. I would appreciate your feedback when convenient."),

    ("professional_email",
     "i would like to request an update regarding my application",
     "I would like to request an update regarding my application. Please let me know if there has been any progress."),

    ("professional_email",
     "thank you for taking the time to speak with me",
     "Thank you for taking the time to speak with me. I appreciate the opportunity to discuss the matter with you."),

    ("professional_email",
     "please let me know a suitable time for a call",
     "Please let me know a suitable time for a call at your convenience."),

    ("professional_email",
     "i have received your email",
     "I have received your email. Thank you for sharing the information."),

    ("professional_email",
     "i look forward to hearing from you",
     "I look forward to hearing from you and appreciate your time and consideration."),

    ("professional_email",
     "can you please confirm whether my application was received",
     "Could you please confirm whether my application has been received successfully?"),

    ("professional_email",
     "i would appreciate an opportunity to discuss my qualifications",
     "I would appreciate the opportunity to discuss my qualifications and how they align with the position."),


    # ======================================
    # PROFESSIONAL MESSAGES
    # ======================================

    ("professional_message",
     "please tell me when you reach",
     "Please let me know once you arrive."),

    ("professional_message",
     "i will call you later",
     "I will get in touch with you later. Please let me know if there is a preferred time."),

    ("professional_message",
     "can you call me when you are available",
     "Could you please call me when you are available?"),

    ("professional_message",
     "i have reached the location",
     "I have arrived at the location."),

    ("professional_message",
     "i am stuck in traffic and will be late",
     "I am currently delayed due to traffic and may arrive later than expected. I apologize for the inconvenience."),

    ("professional_message",
     "please check the file i sent",
     "Could you please review the file I sent when you have a chance?"),

    ("professional_message",
     "i will send the updated file soon",
     "I will share the updated file shortly."),

    ("professional_message",
     "can we discuss this tomorrow",
     "Would it be possible to discuss this tomorrow? Please let me know a convenient time."),

    ("professional_message",
     "thanks for informing me",
     "Thank you for keeping me informed."),

    ("professional_message",
     "i have received the details",
     "I have received the details. Thank you for sharing them."),

    ("professional_message",
     "please remind me about the meeting",
     "Could you please remind me about the meeting when convenient?"),

    ("professional_message",
     "i will be unavailable for some time",
     "I will be unavailable for some time. I will get back to you once I am available."),

    ("professional_message",
     "can you please check this once",
     "Could you please take a look at this when you have a moment?"),

    ("professional_message",
     "i have completed everything from my side",
     "I have completed everything required from my side. Please let me know if anything further is needed."),

    ("professional_message",
     "please give me some time to check",
     "Please allow me some time to review it. I will get back to you shortly."),

    ("professional_message",
     "i am available now",
     "I am available now if you would like to discuss this."),

    ("professional_message",
     "let me know if there is any issue",
     "Please let me know if you encounter any issues."),

    ("professional_message",
     "i will join the meeting shortly",
     "I will join the meeting shortly."),

    ("professional_message",
     "can you send that again",
     "Could you please resend that when convenient?"),

    ("professional_message",
     "i have a question about this",
     "I have a question regarding this. Could you please clarify it for me?"),

    ("professional_message",
     "please update me about the progress",
     "Please keep me updated regarding the progress."),

    ("professional_message",
     "i will get back to you after checking",
     "I will review it and get back to you shortly."),

    ("professional_message",
     "sorry for the inconvenience",
     "I apologize for any inconvenience caused."),

    ("professional_message",
     "thank you for your help",
     "Thank you for your assistance and support."),

    ("professional_message",
     "can we talk about this later",
     "Would it be possible to discuss this later? Please let me know when you are available."),

    ("professional_message",
     "i have shared the details with the team",
     "I have shared the details with the team for their reference."),

    ("professional_message",
     "please let me know if you have received it",
     "Please let me know if you have received it successfully."),

    ("professional_message",
     "i am waiting for your confirmation",
     "I am awaiting your confirmation. Please let me know when convenient."),

    ("professional_message",
     "i will be there on time",
     "I will be there on time as scheduled."),

    ("professional_message",
     "please check your email",
     "Could you please check your email when you have a moment?"),

    ("professional_message",
     "i have shared the meeting details",
     "I have shared the meeting details for your reference."),

    ("professional_message",
     "let me know when we can start",
     "Please let me know when we can begin."),

    ("professional_message",
     "i will keep you posted",
     "I will keep you updated on any further developments."),


    # ======================================
    # ACADEMIC COMMUNICATION
    # ======================================

    ("academic_communication",
     "sir can you explain this topic again",
     "Sir, could you please explain this topic once again? I would appreciate some additional clarification."),

    ("academic_communication",
     "sir when should we submit the project",
     "Sir, could you please confirm the project submission deadline?"),

    ("academic_communication",
     "our team has completed the first phase",
     "Sir, our team has completed the first phase of the project. We would appreciate your feedback on our progress."),

    ("academic_communication",
     "sir can you check our presentation",
     "Sir, could you please review our presentation and let us know if any improvements are required?"),

    ("academic_communication",
     "i could not attend the lecture",
     "I was unable to attend the lecture. Could you please let me know if there were any important announcements or materials I should review?"),

    ("academic_communication",
     "sir can you tell us what we need to prepare",
     "Sir, could you please let us know what we need to prepare for the upcoming session?"),

    ("academic_communication",
     "we have completed the documentation",
     "Our team has completed the required documentation. Kindly let us know if any revisions are needed."),

    ("academic_communication",
     "sir can we discuss our project progress",
     "Sir, could we schedule a brief meeting to discuss our project progress and receive your feedback?"),

    ("academic_communication",
     "please tell me if my assignment needs changes",
     "Could you please review my assignment and let me know if any changes are required?"),

    ("academic_communication",
     "sir i have submitted the file",
     "Sir, I have submitted the required file. Kindly confirm if it has been received successfully."),

    ("academic_communication",
     "when will the viva be conducted",
     "Could you please let us know when the viva is scheduled to be conducted?"),

    ("academic_communication",
     "sir can we change our project topic",
     "Sir, we would like to discuss the possibility of changing our project topic. Could we please seek your guidance on this?"),

    ("academic_communication",
     "our team is facing an issue with the implementation",
     "Our team is currently facing an issue with the implementation. We would appreciate your guidance on how to proceed."),

    ("academic_communication",
     "sir please tell us what changes are needed",
     "Sir, could you please let us know what changes are required so that we can update the project accordingly?"),

    ("academic_communication",
     "can we meet after class",
     "Would it be possible to meet briefly after class to discuss the project?"),

    ("academic_communication",
     "i need clarification regarding the assignment",
     "I would appreciate some clarification regarding the assignment requirements."),

    ("academic_communication",
     "sir can you approve our project idea",
     "Sir, could you please review our project idea and let us know if we may proceed with it?"),

    ("academic_communication",
     "we have made the changes you suggested",
     "We have incorporated the changes you suggested. Kindly review the updated version and let us know if further revisions are needed."),

    ("academic_communication",
     "sir can we submit the report tomorrow",
     "Sir, would it be possible for us to submit the report tomorrow? We would appreciate your consideration."),

    ("academic_communication",
     "please let me know the format for the report",
     "Could you please let me know the required format for the report?"),

    ("academic_communication",
     "our project demonstration is ready",
     "Our project demonstration is ready. Please let us know a suitable time for us to present it."),

    ("academic_communication",
     "sir i have a doubt regarding the evaluation",
     "Sir, I have a question regarding the evaluation criteria. Could you please clarify it?"),

    ("academic_communication",
     "can you please share the study material",
     "Could you please share the relevant study material for the topic?"),

    ("academic_communication",
     "sir we need your approval before proceeding",
     "Sir, we would appreciate your approval before proceeding with the next stage of the project."),

    ("academic_communication",
     "we have divided the work among the team members",
     "We have divided the project responsibilities among the team members and started working accordingly."),

    ("academic_communication",
     "sir can you review our final report",
     "Sir, could you please review our final report and let us know if any corrections are required?"),

    ("academic_communication",
     "i want to know my project evaluation status",
     "I would like to inquire about the current status of my project evaluation."),

    ("academic_communication",
     "sir please let us know if the meeting is confirmed",
     "Sir, could you please confirm whether the meeting is scheduled as planned?"),

    ("academic_communication",
     "we need some guidance regarding our research work",
     "We would appreciate your guidance regarding our research work and the next steps we should follow."),

    ("academic_communication",
     "sir can we get feedback before final submission",
     "Sir, could we please receive your feedback before the final submission so that we can make any necessary improvements?"),

    ("academic_communication",
     "i have completed my part of the project",
     "I have completed my assigned part of the project. Please let me know if any further work is required from my side."),

    ("academic_communication",
     "sir can you confirm the presentation date",
     "Sir, could you please confirm the date scheduled for the project presentation?"),


    # ======================================
    # CORPORATE COMMUNICATION
    # ======================================

    ("corporate_communication",
     "i need leave tomorrow",
     "I would like to request leave for tomorrow. Please let me know if any additional information is required."),

    ("corporate_communication",
     "i will finish the task by evening",
     "I expect to complete the task by the end of the day and will provide an update once it is finished."),

    ("corporate_communication",
     "can you assign me another task",
     "Please let me know if there are any additional tasks I can take up."),

    ("corporate_communication",
     "i have completed the changes requested by the client",
     "I have completed the changes requested by the client. Please review them and let me know if any further modifications are required."),

    ("corporate_communication",
     "the issue has been fixed",
     "The reported issue has been resolved. Please let me know if you encounter any further problems."),

    ("corporate_communication",
     "i need access to the project repository",
     "Could you please provide me with access to the project repository so that I can proceed with the assigned work?"),

    ("corporate_communication",
     "i have updated the documentation",
     "I have updated the documentation with the required changes. Please review it when convenient."),

    ("corporate_communication",
     "can we have a quick meeting",
     "Would it be possible to schedule a brief meeting to discuss this matter?"),

    ("corporate_communication",
     "i will be working from home today",
     "I would like to inform you that I will be working from home today."),

    ("corporate_communication",
     "please tell me if there are any changes",
     "Please let me know if there are any changes or additional requirements."),

    ("corporate_communication",
     "i have shared the weekly update",
     "I have shared the weekly update for your review."),

    ("corporate_communication",
     "the task is taking longer than expected",
     "The task is taking longer than initially expected. I will provide an updated timeline as soon as possible."),

    ("corporate_communication",
     "i need clarification about my responsibilities",
     "I would appreciate some clarification regarding my responsibilities for this task."),

    ("corporate_communication",
     "the client has requested some changes",
     "The client has requested several changes. I will review the requirements and coordinate the necessary updates."),

    ("corporate_communication",
     "i am available to take up this task",
     "I am available to take up this task. Please share the relevant details and requirements."),

    ("corporate_communication",
     "please review the latest build",
     "Could you please review the latest build and let me know if you identify any issues?"),

    ("corporate_communication",
     "i have pushed the latest changes",
     "I have pushed the latest changes to the repository. Please review them when convenient."),

    ("corporate_communication",
     "can you share the project requirements",
     "Could you please share the project requirements so that I can proceed accordingly?"),

    ("corporate_communication",
     "i will need some additional time",
     "I will need some additional time to complete the task properly. I will provide an updated completion estimate shortly."),

    ("corporate_communication",
     "thank you for the feedback",
     "Thank you for your feedback. I will take it into consideration and make the necessary improvements."),

    ("corporate_communication",
     "i have completed the assigned work",
     "I have completed the assigned work. Please let me know if you would like me to make any further changes."),

    ("corporate_communication",
     "can you explain the priority of this task",
     "Could you please clarify the priority of this task so that I can plan my work accordingly?"),

    ("corporate_communication",
     "i am facing a technical issue",
     "I am currently facing a technical issue with the task. I am investigating it and will provide an update shortly."),

    ("corporate_communication",
     "the meeting has been completed",
     "The meeting has been completed. I will follow up on the discussed action items accordingly."),

    ("corporate_communication",
     "please let me know the expected deadline",
     "Could you please confirm the expected deadline for this task?"),

    ("corporate_communication",
     "i have prepared the presentation",
     "I have prepared the presentation for the upcoming discussion. Please let me know if any changes are required."),

    ("corporate_communication",
     "i will share the report by tomorrow",
     "I will share the completed report by tomorrow."),

    ("corporate_communication",
     "please let me know if i should attend the meeting",
     "Please let me know whether my attendance is required for the meeting."),

    ("corporate_communication",
     "i would like to discuss my current responsibilities",
     "I would like to discuss my current responsibilities and clarify a few points regarding my role."),

    ("corporate_communication",
     "i have received the task requirements",
     "I have received the task requirements and will proceed accordingly."),

    ("corporate_communication",
     "please let me know if the client has responded",
     "Please let me know if there has been any response from the client."),

    ("corporate_communication",
     "i will provide the update after testing",
     "I will provide an update once the testing has been completed."),


    # ======================================
    # TONE CORRECTION
    # ======================================

    ("tone_correction",
     "you need to fix this immediately",
     "Could you please review this and make the necessary corrections at your earliest convenience?"),

    ("tone_correction",
     "why haven't you sent the file yet",
     "I wanted to follow up regarding the file, as I have not received it yet. Please let me know when it will be available."),

    ("tone_correction",
     "this doesn't make any sense",
     "I am having difficulty understanding this. Could you please provide some clarification?"),

    ("tone_correction",
     "you are late again",
     "I noticed that you arrived later than scheduled. Please try to be on time for future commitments."),

    ("tone_correction",
     "give me the information now",
     "Could you please provide the information when you have a chance?"),

    ("tone_correction",
     "you made a mistake in the report",
     "I noticed a possible error in the report. Could you please review that section and make the necessary correction?"),

    ("tone_correction",
     "i don't like this idea",
     "I have some concerns regarding this approach. Could we discuss possible alternatives?"),

    ("tone_correction",
     "you should have told me earlier",
     "It would have been helpful to receive this information earlier. Please keep me informed of similar updates in the future."),

    ("tone_correction",
     "this is not acceptable",
     "I have some concerns regarding the current situation. Could we discuss how it can be addressed?"),

    ("tone_correction",
     "stop sending incomplete work",
     "Please ensure that the work is complete before sharing it for review."),

    ("tone_correction",
     "you are not understanding what i am saying",
     "I may not have explained the matter clearly. Let me clarify my point further."),

    ("tone_correction",
     "why did you do it like this",
     "Could you please explain the reasoning behind this approach?"),

    ("tone_correction",
     "you need to respond faster",
     "I would appreciate it if you could respond at your earliest convenience when possible."),

    ("tone_correction",
     "this deadline is impossible",
     "The current deadline may be challenging to meet. Could we discuss the timeline and explore possible alternatives?"),

    ("tone_correction",
     "you forgot to send the attachment",
     "It appears that the attachment may have been missed. Could you please send it when convenient?"),

    ("tone_correction",
     "don't make this mistake again",
     "Please take care to avoid this issue in future submissions."),

    ("tone_correction",
     "i want an answer right now",
     "I would appreciate an update as soon as possible."),

    ("tone_correction",
     "you didn't follow my instructions",
     "It appears that some of the instructions may not have been followed. Could you please review them and make the necessary adjustments?"),

    ("tone_correction",
     "this presentation is bad",
     "There are a few areas of the presentation that could be improved. I would be happy to discuss the suggested changes."),

    ("tone_correction",
     "you need to explain this properly",
     "Could you please provide some additional explanation so that the details are clearer?"),

    ("tone_correction",
     "why are you asking me this",
     "Could you please clarify why this information is required?"),

    ("tone_correction",
     "you are wasting my time",
     "I would appreciate it if we could keep the discussion focused so that we can resolve the matter efficiently."),

    ("tone_correction",
     "send it correctly this time",
     "Could you please review the document carefully before sending the revised version?"),

    ("tone_correction",
     "you should have checked this",
     "It would have been helpful to review this before submission. Please ensure it is checked carefully going forward."),

    ("tone_correction",
     "i don't want to wait anymore",
     "I would appreciate an update at your earliest convenience, as I am waiting for the next steps."),

    ("tone_correction",
     "you are making this more complicated",
     "Could we simplify the approach to make the process more efficient?"),

    ("tone_correction",
     "this is your responsibility",
     "This task falls within your responsibilities. Please let me know if you need any clarification or support."),

    ("tone_correction",
     "don't forget to do this",
     "Please remember to complete this task when you have an opportunity."),

    ("tone_correction",
     "i need you to fix this now",
     "Could you please address this issue at your earliest convenience?"),

    ("tone_correction",
     "you didn't do what i asked",
     "It appears that the requested changes may not have been incorporated. Could you please review the requirements again?"),

    ("tone_correction",
     "this is completely wrong",
     "There appear to be some significant differences from the expected result. Could you please review the work and make the necessary corrections?"),

    ("tone_correction",
     "why haven't you completed this",
     "Could you please provide an update on the status of this task and let me know when it is expected to be completed?"),

    ("tone_correction",
     "you need to listen carefully",
     "Please consider the details carefully so that we can ensure everything is aligned with the requirements."),

]

# ==========================================
# 5. COMBINE WITH EXISTING DATA
# ==========================================

existing_user_prompts = {
    record["messages"][1]["content"].strip().lower()
    for record in existing_records
}

added = 0

for category, user_text, assistant_text in new_examples:

    key = user_text.strip().lower()

    if key in existing_user_prompts:
        continue

    existing_records.append({
        "category": category,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            },
            {
                "role": "assistant",
                "content": assistant_text
            }
        ]
    })

    existing_user_prompts.add(key)
    added += 1


# ==========================================
# 6. SAVE UPDATED DATASET
# ==========================================

with open(jsonl_path, "w", encoding="utf-8") as f:

    for record in existing_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


# ==========================================
# 7. VALIDATE
# ==========================================

with open(jsonl_path, "r", encoding="utf-8") as f:

    final_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("=" * 50)
print("DATASET EXPANSION COMPLETE")
print("=" * 50)

print("Previously existing:", len(existing_records) - added)
print("New examples added:", added)
print("Final dataset size:", len(final_records))

print("\nCATEGORY DISTRIBUTION:")

categories = {}

for record in final_records:

    category = record["category"]
    categories[category] = categories.get(category, 0) + 1

for category, count in sorted(categories.items()):
    print(f"{category}: {count}")

print("\nSaved to:")
print(jsonl_path)

print("\nDataset file verified successfully!")

Existing examples: 39
DATASET EXPANSION COMPLETE
Previously existing: 39
New examples added: 163
Final dataset size: 202

CATEGORY DISTRIBUTION:
academic_communication: 41
corporate_communication: 39
professional_email: 42
professional_message: 41
tone_correction: 39

Saved to:
/content/Professional_Response_Generator/dataset/professional_responses.jsonl

Dataset file verified successfully!


In [17]:
import os
import json
import random
from collections import Counter

# ==========================================
# STEP 5: DATASET QUALITY AND SPLITTING
# ==========================================

DATA_DIR = "/content/Professional_Response_Generator/dataset"

jsonl_path = os.path.join(
    DATA_DIR,
    "professional_responses.jsonl"
)

# Load dataset
with open(jsonl_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("Loaded examples:", len(records))

# ==========================================
# 1. QUALITY VALIDATION
# ==========================================

assert len(records) == 202, "Unexpected dataset size!"

seen_prompts = set()
duplicates = []
invalid_records = []

for index, record in enumerate(records):

    messages = record.get("messages", [])

    if len(messages) != 3:
        invalid_records.append(index)
        continue

    if not all(
        message.get("content", "").strip()
        for message in messages
    ):
        invalid_records.append(index)
        continue

    if [m["role"] for m in messages] != [
        "system", "user", "assistant"
    ]:
        invalid_records.append(index)
        continue

    prompt = messages[1]["content"].strip().lower()

    if prompt in seen_prompts:
        duplicates.append(prompt)

    seen_prompts.add(prompt)

assert not invalid_records, f"Invalid records: {invalid_records}"
assert not duplicates, f"Duplicate prompts: {duplicates}"

print("\nQUALITY CHECK PASSED")
print("Invalid records:", len(invalid_records))
print("Duplicate prompts:", len(duplicates))

# ==========================================
# 2. CATEGORY VALIDATION
# ==========================================

category_counts = Counter(
    record["category"] for record in records
)

print("\nCATEGORY COUNTS:")

for category, count in sorted(category_counts.items()):
    print(f"{category}: {count}")

assert len(category_counts) == 5

# ==========================================
# 3. STRATIFIED 90:10 SPLIT
# ==========================================

random.seed(42)

train_records = []
val_records = []

for category in sorted(category_counts):

    category_records = [
        r for r in records
        if r["category"] == category
    ]

    random.shuffle(category_records)

    val_count = round(len(category_records) * 0.10)

    val_records.extend(category_records[:val_count])
    train_records.extend(category_records[val_count:])

random.shuffle(train_records)
random.shuffle(val_records)

# ==========================================
# 4. SAVE TRAINING AND VALIDATION FILES
# ==========================================

train_path = os.path.join(DATA_DIR, "train.jsonl")
val_path = os.path.join(DATA_DIR, "validation.jsonl")

def save_jsonl(path, data):

    with open(path, "w", encoding="utf-8") as f:

        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(train_path, train_records)
save_jsonl(val_path, val_records)

# ==========================================
# 5. VERIFY SPLIT
# ==========================================

with open(train_path, "r", encoding="utf-8") as f:
    train_check = [json.loads(line) for line in f]

with open(val_path, "r", encoding="utf-8") as f:
    val_check = [json.loads(line) for line in f]

assert len(train_check) + len(val_check) == len(records)

train_prompts = {
    r["messages"][1]["content"].strip().lower()
    for r in train_check
}

val_prompts = {
    r["messages"][1]["content"].strip().lower()
    for r in val_check
}

assert not train_prompts.intersection(val_prompts)

# ==========================================
# 6. DISPLAY RESULTS
# ==========================================

print("\n" + "=" * 45)
print("DATASET SPLIT COMPLETED")
print("=" * 45)

print("Total examples:", len(records))
print("Training examples:", len(train_check))
print("Validation examples:", len(val_check))

print("\nTRAINING CATEGORY DISTRIBUTION:")
print(Counter(r["category"] for r in train_check))

print("\nVALIDATION CATEGORY DISTRIBUTION:")
print(Counter(r["category"] for r in val_check))

print("\nFILES CREATED:")
print(train_path)
print(val_path)

print("\nDATASET SPLIT AND VERIFICATION SUCCESSFUL!")

Loaded examples: 202

QUALITY CHECK PASSED
Invalid records: 0
Duplicate prompts: 0

CATEGORY COUNTS:
academic_communication: 41
corporate_communication: 39
professional_email: 42
professional_message: 41
tone_correction: 39

DATASET SPLIT COMPLETED
Total examples: 202
Training examples: 182
Validation examples: 20

TRAINING CATEGORY DISTRIBUTION:
Counter({'professional_email': 38, 'professional_message': 37, 'academic_communication': 37, 'tone_correction': 35, 'corporate_communication': 35})

VALIDATION CATEGORY DISTRIBUTION:
Counter({'professional_message': 4, 'tone_correction': 4, 'corporate_communication': 4, 'academic_communication': 4, 'professional_email': 4})

FILES CREATED:
/content/Professional_Response_Generator/dataset/train.jsonl
/content/Professional_Response_Generator/dataset/validation.jsonl

DATASET SPLIT AND VERIFICATION SUCCESSFUL!


In [18]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Check GPU
assert torch.cuda.is_available(), "GPU is not enabled!"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("\nTokenizer loaded successfully!")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.config.use_cache = False

print("\nMODEL LOADED SUCCESSFULLY!")
print("Model:", MODEL_NAME)
print("Device:", model.device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

GPU: Tesla T4
CUDA: 13.0


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


MODEL LOADED SUCCESSFULLY!
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
GPU memory allocated: 1.07 GB
GPU memory reserved: 1.12 GB


In [19]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Prepare quantized model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)

# Display trainable parameters
model.print_trainable_parameters()

print("\nQLoRA preparation completed!")

# Verify trainable parameters exist
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

assert trainable > 0, "No trainable parameters found!"

print("Trainable parameter count:", trainable)

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945

QLoRA preparation completed!
Trainable parameter count: 9232384


In [20]:
import json
from datasets import Dataset

DATA_DIR = "/content/Professional_Response_Generator/dataset"

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_jsonl(f"{DATA_DIR}/train.jsonl")
validation_records = load_jsonl(f"{DATA_DIR}/validation.jsonl")

def format_example(example):
    # Apply Qwen's native chat template
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list([
    format_example(example) for example in train_records
])

validation_dataset = Dataset.from_list([
    format_example(example) for example in validation_records
])

print("DATASET FORMATTING COMPLETED")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

print("\nSAMPLE FORMATTED EXAMPLE:\n")
print(train_dataset[0]["text"])

# Verify formatting
assert len(train_dataset) == 182
assert len(validation_dataset) == 20
assert train_dataset[0]["text"].strip()

print("\nDataset verification passed!")

DATASET FORMATTING COMPLETED
Training examples: 182
Validation examples: 20

SAMPLE FORMATTED EXAMPLE:

<|im_start|>system
You are a professional communication assistant. Rewrite informal or rough messages into clear, polished, natural professional communication. Preserve the original meaning, do not invent facts, and use an appropriate tone.<|im_end|>
<|im_start|>user
i am writing regarding the opportunity discussed earlier<|im_end|>
<|im_start|>assistant
I am writing regarding the opportunity we discussed earlier and would appreciate any further information you can provide.<|im_end|>


Dataset verification passed!


In [22]:

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import inspect

# Check supported arguments in your installed version
supported_args = inspect.signature(TrainingArguments.__init__).parameters

print("Checking TrainingArguments compatibility...")

# Build configuration using supported arguments only
config = {
    "output_dir": "/content/Professional_Response_Generator/results",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "logging_steps": 5,
    "fp16": True,
    "gradient_checkpointing": True,
    "report_to": "none",
    "seed": 42
}

# Add optional arguments only when supported
optional_args = {
    "weight_decay": 0.01,
    "warmup_steps": 5,
    "eval_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "optim": "adamw_torch"
}

for key, value in optional_args.items():
    if key in supported_args:
        config[key] = value

# Avoid passing both evaluation strategy aliases
if "eval_strategy" in config:
    config.pop("evaluation_strategy", None)

training_args = TrainingArguments(**config)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator
)

print("\nTRAINER INITIALIZED SUCCESSFULLY!")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Training batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Evaluation strategy:", getattr(training_args, "eval_strategy", "version dependent"))

Checking TrainingArguments compatibility...

TRAINER INITIALIZED SUCCESSFULLY!
Epochs: 3
Learning rate: 0.0002
Training batch size: 2
Gradient accumulation: 4
Evaluation strategy: IntervalStrategy.EPOCH


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [23]:
import torch
import gc

# Clear unused GPU memory
gc.collect()
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Allocated memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved memory:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

print("\nStarting QLoRA fine-tuning...")

# Start training
training_result = trainer.train()

print("\n" + "=" * 50)
print("FINE-TUNING COMPLETED SUCCESSFULLY!")
print("=" * 50)

print("\nTraining metrics:")
for key, value in training_result.metrics.items():
    print(f"{key}: {value}")

GPU: Tesla T4
Allocated memory: 1.54 GB
Reserved memory: 2.02 GB

Starting QLoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.543789,0.490094
2,0.407734,0.441559
3,0.338471,0.427459



FINE-TUNING COMPLETED SUCCESSFULLY!

Training metrics:
train_runtime: 205.3628
train_samples_per_second: 2.659
train_steps_per_second: 0.336
total_flos: 340909043337216.0
train_loss: 0.7956641014071478
epoch: 3.0


In [24]:
import os
import torch

MODEL_OUTPUT = "/content/Professional_Response_Generator/fine_tuned_model"

os.makedirs(MODEL_OUTPUT, exist_ok=True)

# Save trained LoRA adapter
model.save_pretrained(MODEL_OUTPUT)

# Save tokenizer
tokenizer.save_pretrained(MODEL_OUTPUT)

# Save training metrics
import json

metrics_path = os.path.join(MODEL_OUTPUT, "training_metrics.json")

with open(metrics_path, "w") as f:
    json.dump(training_result.metrics, f, indent=4)

print("=" * 50)
print("MODEL SAVED SUCCESSFULLY!")
print("=" * 50)

print("Model location:", MODEL_OUTPUT)
print("\nSaved files:")

for filename in os.listdir(MODEL_OUTPUT):
    print("-", filename)

print("\nAdapter configuration:")
print("LoRA rank:", model.peft_config["default"].r)
print("LoRA alpha:", model.peft_config["default"].lora_alpha)

MODEL SAVED SUCCESSFULLY!
Model location: /content/Professional_Response_Generator/fine_tuned_model

Saved files:
- tokenizer_config.json
- chat_template.jinja
- README.md
- adapter_config.json
- adapter_model.safetensors
- training_metrics.json
- tokenizer.json

Adapter configuration:
LoRA rank: 8
LoRA alpha: 16


In [26]:
import torch

def generate_response(prompt, max_new_tokens=150):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional communication assistant. "
                "Rewrite informal or rough messages into clear, polished, "
                "natural professional communication. Preserve the original "
                "meaning, do not invent facts, and use an appropriate tone."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Generate the formatted conversation
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize explicitly and extract input_ids
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        padding=False
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Generate response
    model.eval()

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[0][input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# Evaluation prompts
test_prompts = [
    "hey sir i wanted to ask if you checked my project report",
    "i cant attend the meeting today because something came up",
    "please send me the project files as soon as possible",
    "sorry for late reply i was busy with college work",
    "i need an update about my internship application"
]

print("=" * 65)
print("FINE-TUNED MODEL EVALUATION")
print("=" * 65)

fine_tuned_results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTEST {i}")
    print("INPUT:", prompt)

    response = generate_response(prompt)

    fine_tuned_results.append({
        "input": prompt,
        "response": response
    })

    print("GENERATED RESPONSE:", response)
    print("-" * 65)

print("\nEvaluation generation completed!")

FINE-TUNED MODEL EVALUATION

TEST 1
INPUT: hey sir i wanted to ask if you checked my project report
GENERATED RESPONSE: Sir, I would like to inquire whether you have reviewed my project report.
-----------------------------------------------------------------

TEST 2
INPUT: i cant attend the meeting today because something came up
GENERATED RESPONSE: I am unable to attend the meeting today due to unforeseen circumstances. I apologize for any inconvenience caused.
-----------------------------------------------------------------

TEST 3
INPUT: please send me the project files as soon as possible
GENERATED RESPONSE: Could you please share the project files so that we can proceed with our work?
-----------------------------------------------------------------

TEST 4
INPUT: sorry for late reply i was busy with college work
GENERATED RESPONSE: I apologize for the delay in my response. I appreciate your understanding. I have been working on some college assignments during this time. Please 

In [27]:
import torch
import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Release the current fine-tuned model from GPU
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

print("Loading original Qwen base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

base_model.eval()

print("Original base model loaded successfully!")

def generate_base_response(prompt, max_new_tokens=150):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional communication assistant. "
                "Rewrite informal or rough messages into clear, polished, "
                "natural professional communication. Preserve the original "
                "meaning, do not invent facts, and use an appropriate tone."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(base_model.device)
    attention_mask = inputs["attention_mask"].to(base_model.device)

    with torch.inference_mode():
        outputs = base_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


base_results = []

print("\n" + "=" * 65)
print("ORIGINAL QWEN BASE MODEL EVALUATION")
print("=" * 65)

for i, prompt in enumerate(test_prompts, 1):

    response = generate_base_response(prompt)

    base_results.append({
        "input": prompt,
        "response": response
    })

    print(f"\nTEST {i}")
    print("INPUT:", prompt)
    print("BASE MODEL RESPONSE:", response)
    print("-" * 65)

print("\nBASE MODEL EVALUATION COMPLETED!")

Loading original Qwen base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Original base model loaded successfully!

ORIGINAL QWEN BASE MODEL EVALUATION

TEST 1
INPUT: hey sir i wanted to ask if you checked my project report
BASE MODEL RESPONSE: Hello,

I hope this message finds you well.

Could you please let me know if you have reviewed my project report?

Thank you for your time.

Best regards,
[Your Name]
-----------------------------------------------------------------

TEST 2
INPUT: i cant attend the meeting today because something came up
BASE MODEL RESPONSE: I'm sorry to hear that you couldn't make it to the meeting. Please let me know if there is anything I can assist with regarding this situation.
-----------------------------------------------------------------

TEST 3
INPUT: please send me the project files as soon as possible
BASE MODEL RESPONSE: Please provide the project files as soon as they become available.
-----------------------------------------------------------------

TEST 4
INPUT: sorry for late reply i was busy with college work
BASE 

In [28]:
import pandas as pd
import os

comparison_data = []

# Preliminary human-assessed scores
base_scores = [17, 11, 16, 18, 10]
fine_scores = [19, 19, 16, 15, 17]

for i in range(len(test_prompts)):
    comparison_data.append({
        "Test": i + 1,
        "Input": test_prompts[i],
        "Base Model Response": base_results[i]["response"],
        "Fine Tuned Response": fine_tuned_results[i]["response"],
        "Base Score": base_scores[i],
        "Fine Tuned Score": fine_scores[i],
        "Improvement": fine_scores[i] - base_scores[i]
    })

comparison_df = pd.DataFrame(comparison_data)

output_path = (
    "/content/Professional_Response_Generator/"
    "model_comparison.csv"
)

comparison_df.to_csv(output_path, index=False)

print("COMPARISON REPORT CREATED!")
print("Location:", output_path)

print("\nAVERAGE SCORES")
print("Base model:", sum(base_scores) / len(base_scores), "/20")
print("Fine-tuned:", sum(fine_scores) / len(fine_scores), "/20")

print("\nOVERALL SCORE")
print("Base:", sum(base_scores), "/100")
print("Fine-tuned:", sum(fine_scores), "/100")

print("\nSaved comparison file successfully!")

COMPARISON REPORT CREATED!
Location: /content/Professional_Response_Generator/model_comparison.csv

AVERAGE SCORES
Base model: 14.4 /20
Fine-tuned: 17.2 /20

OVERALL SCORE
Base: 72 /100
Fine-tuned: 86 /100

Saved comparison file successfully!


In [29]:
import json
from collections import Counter

DATA_DIR = "/content/Professional_Response_Generator/dataset"

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl(f"{DATA_DIR}/train.jsonl")
validation_records = read_jsonl(f"{DATA_DIR}/validation.jsonl")

print("=" * 55)
print("DATASET QUALITY REVIEW")
print("=" * 55)

print("Training examples:", len(train_records))
print("Validation examples:", len(validation_records))

print("\nCATEGORY DISTRIBUTION:")
categories = Counter(
    record.get("category", "unknown")
    for record in train_records
)

for category, count in categories.items():
    print(category, ":", count)

print("\nSAMPLE TRAINING EXAMPLES:")

for i, record in enumerate(train_records[:10], 1):
    print(f"\nEXAMPLE {i}")

    for message in record["messages"]:
        if message["role"] == "user":
            print("INPUT:", message["content"])
        elif message["role"] == "assistant":
            print("OUTPUT:", message["content"])

print("\nDATASET REVIEW COMPLETED!")

DATASET QUALITY REVIEW
Training examples: 182
Validation examples: 20

CATEGORY DISTRIBUTION:
professional_email : 38
tone_correction : 35
professional_message : 37
corporate_communication : 35
academic_communication : 37

SAMPLE TRAINING EXAMPLES:

EXAMPLE 1
INPUT: i am writing regarding the opportunity discussed earlier
OUTPUT: I am writing regarding the opportunity we discussed earlier and would appreciate any further information you can provide.

EXAMPLE 2
INPUT: thank you for your response
OUTPUT: Thank you for your response. I appreciate you taking the time to address my query.

EXAMPLE 3
INPUT: don't make this mistake again
OUTPUT: Please take care to avoid this issue in future submissions.

EXAMPLE 4
INPUT: you need to explain this properly
OUTPUT: Could you please provide some additional explanation so that the details are clearer?

EXAMPLE 5
INPUT: i have shared the meeting details
OUTPUT: I have shared the meeting details for your reference.

EXAMPLE 6
INPUT: i am following 

In [30]:
import json
import os

DATA_DIR = "/content/Professional_Response_Generator/dataset"

improvement_examples = [
    ("professional_message", "hey sir did you check my report", "Sir, could you please let me know if you have reviewed my report?"),
    ("professional_message", "send me the files today", "Could you please share the files today?"),
    ("professional_message", "i cannot come today", "I will not be able to attend today."),
    ("professional_message", "tell me when you are free", "Please let me know when you are available."),
    ("professional_message", "i need your approval", "I would appreciate your approval."),
    ("professional_message", "please reply fast", "Could you please respond at your earliest convenience?"),
    ("professional_message", "i have completed the task", "I have completed the task."),
    ("professional_message", "i will send it tomorrow", "I will share it tomorrow."),
    ("professional_message", "can we talk about this", "Could we discuss this when you are available?"),
    ("professional_message", "i need more time", "I would appreciate some additional time to complete this."),

    ("academic_communication", "sir i have a doubt in this topic", "Sir, I have a question regarding this topic."),
    ("academic_communication", "when is the assignment deadline", "Could you please confirm the assignment deadline?"),
    ("academic_communication", "i submitted my assignment", "I have submitted my assignment."),
    ("academic_communication", "please explain this question", "Could you please explain this question?"),
    ("academic_communication", "i was absent yesterday", "I was unable to attend yesterday."),
    ("academic_communication", "can you share the notes", "Could you please share the notes?"),
    ("academic_communication", "i need an extension for submission", "I would like to request an extension for the submission."),
    ("academic_communication", "when will the results come", "Could you please let me know when the results will be announced?"),
    ("academic_communication", "i have completed my project", "I have completed my project."),
    ("academic_communication", "please check my document", "Could you please review my document?"),

    ("professional_email", "i am applying for this position", "I am writing to express my interest in this position."),
    ("professional_email", "please find my resume attached", "Please find my resume attached for your review."),
    ("professional_email", "i want to know the interview date", "Could you please confirm the interview date?"),
    ("professional_email", "thank you for considering my application", "Thank you for considering my application."),
    ("professional_email", "i am waiting for your response", "I look forward to your response."),
    ("professional_email", "i am interested in the opportunity", "I am interested in this opportunity and would appreciate further details."),
    ("professional_email", "i want to reschedule the meeting", "I would like to request a rescheduling of the meeting."),
    ("professional_email", "i have attached the required documents", "I have attached the required documents for your review."),
    ("professional_email", "please confirm if you received my email", "Could you please confirm receipt of my email?"),
    ("professional_email", "sorry for the delayed response", "I apologize for the delayed response."),

    ("corporate_communication", "the meeting is moved to friday", "The meeting has been rescheduled to Friday."),
    ("corporate_communication", "we finished the project work", "We have completed the project work."),
    ("corporate_communication", "the report is not ready", "The report is not yet ready."),
    ("corporate_communication", "we need to discuss the budget", "We need to discuss the budget."),
    ("corporate_communication", "please update the shared document", "Please update the shared document."),
    ("corporate_communication", "the client wants changes", "The client has requested changes."),
    ("corporate_communication", "we will start tomorrow", "We will begin tomorrow."),
    ("corporate_communication", "there is a delay in delivery", "There is a delay in the delivery."),
    ("corporate_communication", "the team needs more information", "The team requires additional information."),
    ("corporate_communication", "please confirm the final schedule", "Please confirm the final schedule."),

    ("tone_correction", "you are not listening to me", "I would appreciate it if you could consider my concerns."),
    ("tone_correction", "this work is completely wrong", "There are some issues with the current work that need to be addressed."),
    ("tone_correction", "you did not send anything", "I have not received the requested information yet."),
    ("tone_correction", "stop delaying the work", "Could you please help ensure the work progresses without further delays?"),
    ("tone_correction", "this makes no sense", "Could you please clarify this? I am having difficulty understanding it."),
    ("tone_correction", "you forgot to reply", "I wanted to follow up as I have not received a response yet."),
    ("tone_correction", "do it properly this time", "Please ensure the task is completed accurately."),
    ("tone_correction", "i don't agree with this", "I have a different perspective on this matter."),
    ("tone_correction", "you made a mistake", "I noticed a possible error that may need to be reviewed."),
    ("tone_correction", "why haven't you finished", "Could you please share an update on the progress?")
]

# Load original training examples
train_path = f"{DATA_DIR}/train.jsonl"

with open(train_path, "r", encoding="utf-8") as f:
    original_records = [json.loads(line) for line in f if line.strip()]

existing_prompts = {
    m["content"].strip().lower()
    for record in original_records
    for m in record["messages"]
    if m["role"] == "user"
}

new_records = []

for category, user_text, assistant_text in improvement_examples:
    if user_text.strip().lower() in existing_prompts:
        continue

    new_records.append({
        "category": category,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a professional communication assistant. "
                    "Rewrite informal messages into clear, natural, "
                    "professional language. Preserve the original meaning. "
                    "Do not invent facts, reasons, or context. Be concise."
                )
            },
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    })

# Save improved training version separately
improved_path = f"{DATA_DIR}/improved_train.jsonl"

combined_records = original_records + new_records

with open(improved_path, "w", encoding="utf-8") as f:
    for record in combined_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("=" * 50)
print("IMPROVED DATASET CREATED")
print("=" * 50)
print("Original training examples:", len(original_records))
print("New examples added:", len(new_records))
print("Combined training examples:", len(combined_records))
print("Saved at:", improved_path)

assert len(new_records) > 0
print("\nDataset creation successful!")

IMPROVED DATASET CREATED
Original training examples: 182
New examples added: 49
Combined training examples: 231
Saved at: /content/Professional_Response_Generator/dataset/improved_train.jsonl

Dataset creation successful!


In [31]:
import json
import re
from collections import Counter

BASE = "/content/Professional_Response_Generator/dataset"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def get_user_prompt(example):
    for message in example.get("messages", []):
        if message.get("role") == "user":
            return re.sub(r"\s+", " ", message.get("content", "").strip().lower())
    return ""

train = load_jsonl(f"{BASE}/train.jsonl")
validation = load_jsonl(f"{BASE}/validation.jsonl")
improved = load_jsonl(f"{BASE}/improved_train.jsonl")

print("Original training examples:", len(train))
print("Validation examples:", len(validation))
print("Improved dataset before cleaning:", len(improved))

# Preserve the original training examples.
cleaned = list(improved[:len(train)])

# Build prompt sets to prevent overlap and duplicates.
seen_prompts = {get_user_prompt(x) for x in cleaned}
validation_prompts = {get_user_prompt(x) for x in validation}

removed_duplicates = 0
removed_validation_overlap = 0
added_examples_kept = 0

for example in improved[len(train):]:
    prompt = get_user_prompt(example)

    if not prompt:
        removed_duplicates += 1
        continue

    if prompt in validation_prompts:
        removed_validation_overlap += 1
        continue

    if prompt in seen_prompts:
        removed_duplicates += 1
        continue

    cleaned.append(example)
    seen_prompts.add(prompt)
    added_examples_kept += 1

# Check dataset structure.
invalid = []
for i, example in enumerate(cleaned):
    messages = example.get("messages", [])
    roles = {m.get("role") for m in messages}

    if not {"system", "user", "assistant"}.issubset(roles):
        invalid.append(i)

assert not invalid, f"Invalid records found at indexes: {invalid}"

# Save the cleaned dataset.
output_path = f"{BASE}/improved_train.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for example in cleaned:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

category_counts = Counter(x.get("category", "uncategorized") for x in cleaned)

print("\n" + "=" * 48)
print("DATASET VALIDATION COMPLETE")
print("=" * 48)
print("Final training examples:", len(cleaned))
print("New examples retained:", added_examples_kept)
print("Duplicates removed:", removed_duplicates)
print("Validation overlaps removed:", removed_validation_overlap)
print("Invalid records:", len(invalid))
print("\nCategory distribution:")
for category, count in sorted(category_counts.items()):
    print(f"{category}: {count}")

print("\nValidation examples remain separate:", len(validation))
print("Saved:", output_path)

Original training examples: 182
Validation examples: 20
Improved dataset before cleaning: 231

DATASET VALIDATION COMPLETE
Final training examples: 231
New examples retained: 49
Duplicates removed: 0
Validation overlaps removed: 0
Invalid records: 0

Category distribution:
academic_communication: 47
corporate_communication: 45
professional_email: 47
professional_message: 47
tone_correction: 45

Validation examples remain separate: 20
Saved: /content/Professional_Response_Generator/dataset/improved_train.jsonl


In [32]:
import json
from datasets import Dataset

DATASET_PATH = "/content/Professional_Response_Generator/dataset/improved_train.jsonl"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    improved_data = [json.loads(line) for line in f if line.strip()]

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

improved_dataset = Dataset.from_list(improved_data)
formatted_dataset = improved_dataset.map(format_example)

print("VERSION 2 DATASET READY")
print("Training examples:", len(formatted_dataset))
print("\nSample formatted example:\n")
print(formatted_dataset[0]["text"][:1200])

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

VERSION 2 DATASET READY
Training examples: 231

Sample formatted example:

<|im_start|>system
You are a professional communication assistant. Rewrite informal or rough messages into clear, polished, natural professional communication. Preserve the original meaning, do not invent facts, and use an appropriate tone.<|im_end|>
<|im_start|>user
i am writing regarding the opportunity discussed earlier<|im_end|>
<|im_start|>assistant
I am writing regarding the opportunity we discussed earlier and would appreciate any further information you can provide.<|im_end|>



In [33]:
from transformers import DataCollatorForLanguageModeling

MAX_LENGTH = 512

# Ensure the tokenizer has a padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

tokenized_train_dataset = formatted_dataset.map(
    tokenize_example,
    batched=False,
    remove_columns=formatted_dataset.column_names
)

# Calculate token lengths.
token_lengths = [
    len(example["input_ids"])
    for example in tokenized_train_dataset
]

print("=" * 45)
print("VERSION 2 TOKENIZATION COMPLETE")
print("=" * 45)
print("Training examples:", len(tokenized_train_dataset))
print("Maximum sequence length:", max(token_lengths))
print("Average sequence length:", round(sum(token_lengths) / len(token_lengths), 2))
print("Minimum sequence length:", min(token_lengths))
print("Sequences reaching maximum length:", sum(
    length == MAX_LENGTH for length in token_lengths
))

# Prepare the language-modeling data collator.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("\nData collator initialized successfully.")

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

VERSION 2 TOKENIZATION COMPLETE
Training examples: 231
Maximum sequence length: 92
Average sequence length: 73.45
Minimum sequence length: 60
Sequences reaching maximum length: 0

Data collator initialized successfully.


In [34]:
import torch
import inspect
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

# Release any previous training objects, if present.
if "trainer" in globals():
    del trainer

torch.cuda.empty_cache()

# Prepare the original base model for a fresh LoRA adapter.
base_model = prepare_model_for_kbit_training(base_model)

# Same LoRA configuration as Version 1.
lora_config_v2 = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_v2 = get_peft_model(base_model, lora_config_v2)

model_v2.print_trainable_parameters()

# Keep the Version 1 training schedule.
training_config = {
    "output_dir": "/content/Professional_Response_Generator/training_v2",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 5,
    "fp16": True,
    "gradient_checkpointing": True,
    "logging_steps": 5,
    "save_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "report_to": "none",
    "seed": 42
}

# Adapt argument names/options to the installed Transformers version.
supported = inspect.signature(TrainingArguments.__init__).parameters

if "evaluation_strategy" not in supported and "eval_strategy" in supported:
    training_config["eval_strategy"] = training_config.pop("evaluation_strategy")

training_config = {
    key: value for key, value in training_config.items()
    if key in supported
}

training_args_v2 = TrainingArguments(**training_config)

print("\nVERSION 2 TRAINING CONFIGURATION READY")
print("Epochs:", training_args_v2.num_train_epochs)
print("Learning rate:", training_args_v2.learning_rate)
print("Training batch size:", training_args_v2.per_device_train_batch_size)
print("Gradient accumulation:", training_args_v2.gradient_accumulation_steps)
print("GPU:", torch.cuda.get_device_name(0))

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945

VERSION 2 TRAINING CONFIGURATION READY
Epochs: 3
Learning rate: 0.0002
Training batch size: 2
Gradient accumulation: 4
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [36]:
import json
from datasets import Dataset

VALIDATION_PATH = "/content/Professional_Response_Generator/dataset/validation.jsonl"

with open(VALIDATION_PATH, "r", encoding="utf-8") as f:
    validation_data = [json.loads(line) for line in f if line.strip()]

validation_dataset = Dataset.from_list(validation_data)

def format_validation_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

formatted_validation_dataset = validation_dataset.map(
    format_validation_example
)

tokenized_validation_dataset = formatted_validation_dataset.map(
    tokenize_example,
    batched=False,
    remove_columns=formatted_validation_dataset.column_names
)

print("VALIDATION DATASET READY")
print("Validation examples:", len(tokenized_validation_dataset))
print("Sample token length:", len(tokenized_validation_dataset[0]["input_ids"]))

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

VALIDATION DATASET READY
Validation examples: 20
Sample token length: 72


In [37]:
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    data_collator=data_collator
)

print("Trainer initialized successfully!")
print("Training examples:", len(tokenized_train_dataset))
print("Validation examples:", len(tokenized_validation_dataset))

Trainer initialized successfully!
Training examples: 231
Validation examples: 20


In [38]:
print("Starting Version 2 QLoRA training...")
print("Please wait until all 3 epochs are completed.")

training_result_v2 = trainer_v2.train()

print("\n" + "=" * 50)
print("VERSION 2 TRAINING COMPLETED")
print("=" * 50)

print("Training loss:", training_result_v2.training_loss)
print("Training runtime (seconds):",
      training_result_v2.metrics.get("train_runtime"))

print("\nFinal evaluation:")
eval_results_v2 = trainer_v2.evaluate()
print(eval_results_v2)

Starting Version 2 QLoRA training...
Please wait until all 3 epochs are completed.


Epoch,Training Loss,Validation Loss
1,0.505468,0.481237
2,0.386795,0.426354
3,0.328646,0.414156



VERSION 2 TRAINING COMPLETED
Training loss: 0.7261201691353458
Training runtime (seconds): 249.9515

Final evaluation:


Training Loss,Validation Loss,Epoch
0.328646,0.414156,3


{'eval_loss': 0.4141562581062317}


In [39]:
import os
import json

SAVE_PATH_V2 = (
    "/content/Professional_Response_Generator/fine_tuned_model_v2"
)

os.makedirs(SAVE_PATH_V2, exist_ok=True)

# Save LoRA adapter and tokenizer.
model_v2.save_pretrained(SAVE_PATH_V2)
tokenizer.save_pretrained(SAVE_PATH_V2)

# Save training metrics.
metrics_v2 = {
    "version": "Version 2",
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "training_examples": 231,
    "validation_examples": 20,
    "epochs": 3,
    "learning_rate": 0.0002,
    "lora_rank": 8,
    "lora_alpha": 16,
    "final_training_loss": training_result_v2.training_loss,
    "final_validation_loss": eval_results_v2["eval_loss"],
    "training_runtime_seconds": training_result_v2.metrics.get("train_runtime")
}

with open(
    os.path.join(SAVE_PATH_V2, "training_metrics.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics_v2, f, indent=4)

print("=" * 50)
print("VERSION 2 MODEL SAVED SUCCESSFULLY")
print("=" * 50)
print("Location:", SAVE_PATH_V2)
print("Adapter exists:", os.path.exists(
    os.path.join(SAVE_PATH_V2, "adapter_model.safetensors")
))
print("Metrics:", metrics_v2)
print("\nSaved files:", os.listdir(SAVE_PATH_V2))

VERSION 2 MODEL SAVED SUCCESSFULLY
Location: /content/Professional_Response_Generator/fine_tuned_model_v2
Adapter exists: True
Metrics: {'version': 'Version 2', 'base_model': 'Qwen/Qwen2.5-1.5B-Instruct', 'training_examples': 231, 'validation_examples': 20, 'epochs': 3, 'learning_rate': 0.0002, 'lora_rank': 8, 'lora_alpha': 16, 'final_training_loss': 0.7261201691353458, 'final_validation_loss': 0.4141562581062317, 'training_runtime_seconds': 249.9515}

Saved files: ['tokenizer_config.json', 'chat_template.jinja', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'training_metrics.json', 'tokenizer.json']


In [40]:
test_prompts_v2 = [
    "sir i wanted to know if there is any update about my project evaluation",
    "i will not be able to join the call because i have another commitment",
    "send the updated presentation before evening",
    "thanks for helping me with the assignment",
    "i have completed the changes you suggested",
    "can you please tell me when the internship results will be announced",
    "the deadline is too short and we need more time",
    "i did not understand the feedback given during the review",
    "please check the attached file and let me know if anything is missing",
    "i am interested in applying for the position",
    "you have not replied to my previous message",
    "we found an issue during testing and are working on fixing it",
    "i need permission to leave early tomorrow",
    "the meeting has been postponed to another date",
    "i appreciate your time and consideration"
]

print("TEST BENCHMARK CREATED")
print("Total unseen prompts:", len(test_prompts_v2))

for i, prompt in enumerate(test_prompts_v2, 1):
    print(f"{i}. {prompt}")

TEST BENCHMARK CREATED
Total unseen prompts: 15
1. sir i wanted to know if there is any update about my project evaluation
2. i will not be able to join the call because i have another commitment
3. send the updated presentation before evening
4. thanks for helping me with the assignment
5. i have completed the changes you suggested
6. can you please tell me when the internship results will be announced
7. the deadline is too short and we need more time
8. i did not understand the feedback given during the review
9. please check the attached file and let me know if anything is missing
10. i am interested in applying for the position
11. you have not replied to my previous message
12. we found an issue during testing and are working on fixing it
13. i need permission to leave early tomorrow
14. the meeting has been postponed to another date
15. i appreciate your time and consideration


In [41]:
import torch
from peft import PeftModel

VERSION_1_PATH = (
    "/content/Professional_Response_Generator/fine_tuned_model"
)

VERSION_2_PATH = (
    "/content/Professional_Response_Generator/fine_tuned_model_v2"
)

# Attach Version 1 to the existing Version 2 model.
if "version1" not in model_v2.peft_config:
    model_v2.load_adapter(
        VERSION_1_PATH,
        adapter_name="version1"
    )

print("Both adapters loaded:")
print(list(model_v2.peft_config.keys()))

SYSTEM_PROMPT = (
    "You are a professional communication assistant. "
    "Rewrite informal or rough messages into clear, polished, "
    "natural professional communication. Preserve the original "
    "meaning, do not invent facts, and use an appropriate tone."
)

def generate_with_adapter(adapter_name, prompt):
    model_v2.set_adapter(adapter_name)
    model_v2.eval()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model_v2.device)

    with torch.inference_mode():
        output = model_v2.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

# Generate both versions on the same 15 prompts.
benchmark_results = []

for index, prompt in enumerate(test_prompts_v2, 1):
    response_v1 = generate_with_adapter("version1", prompt)
    response_v2 = generate_with_adapter("default", prompt)

    benchmark_results.append({
        "id": index,
        "prompt": prompt,
        "version_1": response_v1,
        "version_2": response_v2
    })

    print(f"\n{'=' * 65}")
    print(f"TEST {index}/15")
    print("INPUT:", prompt)
    print("\nVERSION 1:", response_v1)
    print("\nVERSION 2:", response_v2)

# Return the model to Version 2.
model_v2.set_adapter("default")

print("\n" + "=" * 65)
print("BENCHMARK GENERATION COMPLETED")
print("Total prompts:", len(benchmark_results))

Both adapters loaded:
['default', 'version1']

TEST 1/15
INPUT: sir i wanted to know if there is any update about my project evaluation

VERSION 1: Sir, I would like to inquire about the status of my project evaluation. Could you please let me know when we expect to receive the results?

VERSION 2: Sir, I would like to inquire about the current status of my project evaluation. Could you please let me know when we expect the results?

TEST 2/15
INPUT: i will not be able to join the call because i have another commitment

VERSION 1: I will not be available for the meeting due to other commitments. I apologize for any inconvenience caused.

VERSION 2: I will unfortunately not be able to attend the meeting due to other commitments. I apologize for any inconvenience caused.

TEST 3/15
INPUT: send the updated presentation before evening

VERSION 1: Could you please share the updated version of the presentation by tomorrow afternoon?

VERSION 2: Could you please share the updated version of t

In [42]:
import pandas as pd
import os

benchmark_df = pd.DataFrame(benchmark_results)

benchmark_path = (
    "/content/Professional_Response_Generator/"
    "version1_vs_version2_benchmark.csv"
)

benchmark_df.to_csv(benchmark_path, index=False)

print("BENCHMARK SAVED SUCCESSFULLY")
print("File:", benchmark_path)
print("Total evaluated prompts:", len(benchmark_df))
print("\nPreview:")
display(benchmark_df.head(5))

BENCHMARK SAVED SUCCESSFULLY
File: /content/Professional_Response_Generator/version1_vs_version2_benchmark.csv
Total evaluated prompts: 15

Preview:


,id,prompt,version_1,version_2
0,1,sir i wanted to know if there is any update ab...,"Sir, I would like to inquire about the status ...","Sir, I would like to inquire about the current..."
1,2,i will not be able to join the call because i ...,I will not be available for the meeting due to...,I will unfortunately not be able to attend the...
2,3,send the updated presentation before evening,Could you please share the updated version of ...,Could you please share the updated version of ...
3,4,thanks for helping me with the assignment,Thank you for your assistance in completing th...,Thank you for your assistance in completing my...
4,5,i have completed the changes you suggested,I have completed the changes requested by you....,I have completed the changes requested by you....


In [43]:
import pandas as pd

audit_criteria = [
    "Preserves request for project evaluation update",
    "Preserves inability to attend and reason",
    "Preserves deadline of before evening",
    "Preserves gratitude for assignment help",
    "Preserves completion of suggested changes",
    "Preserves question about internship announcement",
    "Preserves need for additional time",
    "Preserves lack of understanding of feedback",
    "Preserves request to check attached file",
    "Preserves interest in applying for position",
    "Preserves recipient's lack of reply",
    "Preserves issue found during testing and ongoing fix",
    "Preserves request for permission to leave early tomorrow",
    "Preserves meeting postponement without inventing a date",
    "Preserves appreciation for time and consideration"
]

audit_df = benchmark_df[["id", "prompt", "version_1", "version_2"]].copy()

audit_df["expected_meaning"] = audit_criteria
audit_df["v1_meaning_preserved"] = ""
audit_df["v2_meaning_preserved"] = ""
audit_df["v1_invented_information"] = ""
audit_df["v2_invented_information"] = ""
audit_df["review_notes"] = ""

audit_path = (
    "/content/Professional_Response_Generator/"
    "meaning_preservation_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print("MEANING PRESERVATION AUDIT CREATED")
print("Total prompts:", len(audit_df))
print("Audit file:", audit_path)
display(audit_df[[
    "id", "expected_meaning",
    "v1_meaning_preserved", "v2_meaning_preserved"
]])

MEANING PRESERVATION AUDIT CREATED
Total prompts: 15
Audit file: /content/Professional_Response_Generator/meaning_preservation_audit.csv


,id,expected_meaning,v1_meaning_preserved,v2_meaning_preserved
0,1,Preserves request for project evaluation update,,
1,2,Preserves inability to attend and reason,,
2,3,Preserves deadline of before evening,,
3,4,Preserves gratitude for assignment help,,
4,5,Preserves completion of suggested changes,,
5,6,Preserves question about internship announcement,,
6,7,Preserves need for additional time,,
7,8,Preserves lack of understanding of feedback,,
8,9,Preserves request to check attached file,,
9,10,Preserves interest in applying for position,,


In [44]:
import pandas as pd

# Manual qualitative assessment based on the actual generated outputs.
v1_preservation = [
    "Partial", # 1: Adds a question about when results are expected.
    "Partial", # 2: Changes call to meeting.
    "No",      # 3: Changes before evening to tomorrow afternoon.
    "Yes",     # 4: Gratitude preserved.
    "Partial", # 5: Adds a request for further review/adjustments.
    "Yes",     # 6: Internship announcement question preserved.
    "Yes",     # 7: Need for more time preserved.
    "Yes",     # 8: Feedback clarification request preserved.
    "Yes",     # 9: Attached file review request preserved.
    "Partial", # 10: Adds an unsupported claim about team contribution.
    "Partial", # 11: Changes the previous message into a previous conversation.
    "Yes",     # 12: Issue and ongoing resolution preserved.
    "Yes",     # 13: Permission request preserved.
    "Yes",     # 14: Meeting postponement preserved.
    "Partial"  # 15: Refers to an unspecified request.
]

v2_preservation = [
    "Partial", # 1: Adds a question about when results are expected.
    "Partial", # 2: Changes call to meeting.
    "No",      # 3: Changes before evening to tomorrow afternoon.
    "Yes",     # 4: Gratitude preserved.
    "Partial", # 5: Adds a request for review.
    "Yes",     # 6: Internship announcement question preserved.
    "Yes",     # 7: Need for more time preserved.
    "Yes",     # 8: Feedback clarification request preserved.
    "Yes",     # 9: Attached file review request preserved.
    "Partial", # 10: Adds an inquiry about the application process.
    "Partial", # 11: Changes the follow-up wording and adds an expectation.
    "Yes",     # 12: Issue and ongoing resolution preserved.
    "Yes",     # 13: Permission request preserved.
    "Yes",     # 14: Meeting postponement preserved.
    "Partial"  # 15: Refers to an unspecified request.
]

v1_invention = [
    "Yes", "No", "Yes", "No", "Yes",
    "No", "No", "No", "No", "Yes",
    "Yes", "Yes", "No", "Yes", "Yes"
]

v2_invention = [
    "Yes", "No", "Yes", "No", "Yes",
    "No", "No", "No", "No", "Yes",
    "Yes", "No", "No", "Yes", "Yes"
]

review_notes = [
    "Adds an extra question about expected results.",
    "Changes call to meeting, but retains the reason.",
    "Changes the deadline from before evening to tomorrow afternoon.",
    "Gratitude and appreciation are retained.",
    "Adds a review-related request not explicitly stated.",
    "Preserves the question about internship results.",
    "Preserves the request for additional time.",
    "Preserves the need for feedback clarification.",
    "Preserves the attached-file review request.",
    "Adds information about contribution or application procedure.",
    "Changes the wording and context of the previous message.",
    "Preserves the testing issue and ongoing resolution.",
    "Preserves the permission request and date.",
    "Preserves postponement; adds a calendar instruction.",
    "Adds an implied reference to a request."
]

audit_df["v1_meaning_preserved"] = v1_preservation
audit_df["v2_meaning_preserved"] = v2_preservation
audit_df["v1_invented_information"] = v1_invention
audit_df["v2_invented_information"] = v2_invention
audit_df["review_notes"] = review_notes

audit_path = (
    "/content/Professional_Response_Generator/"
    "meaning_preservation_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

# Calculate qualitative preservation counts.
for version, values in [
    ("Version 1", v1_preservation),
    ("Version 2", v2_preservation)
]:
    counts = pd.Series(values).value_counts()
    print(f"\n{version}")
    print("Yes:", counts.get("Yes", 0))
    print("Partial:", counts.get("Partial", 0))
    print("No:", counts.get("No", 0))

print("\nAUDIT COMPLETED")
print("Saved:", audit_path)

display(audit_df[[
    "id",
    "v1_meaning_preserved",
    "v2_meaning_preserved",
    "v1_invented_information",
    "v2_invented_information"
]])


Version 1
Yes: 8
Partial: 6
No: 1

Version 2
Yes: 8
Partial: 6
No: 1

AUDIT COMPLETED
Saved: /content/Professional_Response_Generator/meaning_preservation_audit.csv


,id,v1_meaning_preserved,v2_meaning_preserved,v1_invented_information,v2_invented_information
0,1,Partial,Partial,Yes,Yes
1,2,Partial,Partial,No,No
2,3,No,No,Yes,Yes
3,4,Yes,Yes,No,No
4,5,Partial,Partial,Yes,Yes
5,6,Yes,Yes,No,No
6,7,Yes,Yes,No,No
7,8,Yes,Yes,No,No
8,9,Yes,Yes,No,No
9,10,Partial,Partial,Yes,Yes


In [45]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Clear memory from previous training and evaluation
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Reuse your existing tokenizer if available
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Same 4-bit quantization configuration used during training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load a fresh, unmodified base model
base_eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

base_eval_model.eval()

print("Fresh base model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fresh base model loaded successfully!
GPU: Tesla T4


In [47]:
import pandas as pd
import torch
from tqdm.auto import tqdm

benchmark_path = "/content/Professional_Response_Generator/version1_vs_version2_benchmark.csv"
benchmark_df = pd.read_csv(benchmark_path)

assert len(benchmark_df) == 15

SYSTEM_PROMPT = (
    "You are a professional communication assistant. "
    "Rewrite informal or rough messages into clear, polished, "
    "natural professional communication. Preserve the original "
    "meaning, do not invent facts, and use an appropriate tone."
)

def generate_response(model, prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

base_responses = []

for prompt in tqdm(benchmark_df["prompt"], desc="Base model evaluation"):
    response = generate_response(base_eval_model, prompt)
    base_responses.append(response)

benchmark_df["base_model"] = base_responses
benchmark_df.to_csv(benchmark_path, index=False)

print("\nBASE MODEL EVALUATION COMPLETED")
print("Total prompts evaluated:", len(base_responses))
print("Saved:", benchmark_path)

for i, row in benchmark_df.head(5).iterrows():
    print(f"\nPROMPT {i+1}: {row['prompt']}")
    print("BASE MODEL:", row["base_model"])

Base model evaluation:   0%|          | 0/15 [00:00<?, ?it/s]


BASE MODEL EVALUATION COMPLETED
Total prompts evaluated: 15
Saved: /content/Professional_Response_Generator/version1_vs_version2_benchmark.csv

PROMPT 1: sir i wanted to know if there is any update about my project evaluation
BASE MODEL: Dear Sir,

I would like to inquire about any updates regarding the evaluation of my project.

Thank you for your time and assistance.

Best regards,
[Your Name]

PROMPT 2: i will not be able to join the call because i have another commitment
BASE MODEL: I am unable to participate in the call due to having other commitments at the moment.

PROMPT 3: send the updated presentation before evening
BASE MODEL: Please send the updated presentation by the end of the day.

PROMPT 4: thanks for helping me with the assignment
BASE MODEL: Thank you for using my services to assist with your assignment. I'm glad to have been able to help. If you need any further assistance in the future, please don't hesitate to reach out. Have a great day!

PROMPT 5: i have comple

In [48]:
import pandas as pd
from IPython.display import display

benchmark_path = "/content/Professional_Response_Generator/version1_vs_version2_benchmark.csv"

comparison_df = pd.read_csv(benchmark_path)

required_columns = ["prompt", "base_model", "version_1", "version_2"]
assert all(col in comparison_df.columns for col in required_columns)

print("THREE-MODEL COMPARISON")
print("Total prompts:", len(comparison_df))

display(
    comparison_df[required_columns].style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top"
    })
)

THREE-MODEL COMPARISON
Total prompts: 15


,prompt,base_model,version_1,version_2
0,sir i wanted to know if there is any update about my project evaluation,"Dear Sir, I would like to inquire about any updates regarding the evaluation of my project. Thank you for your time and assistance. Best regards, [Your Name]","Sir, I would like to inquire about the status of my project evaluation. Could you please let me know when we expect to receive the results?","Sir, I would like to inquire about the current status of my project evaluation. Could you please let me know when we expect the results?"
1,i will not be able to join the call because i have another commitment,I am unable to participate in the call due to having other commitments at the moment.,I will not be available for the meeting due to other commitments. I apologize for any inconvenience caused.,I will unfortunately not be able to attend the meeting due to other commitments. I apologize for any inconvenience caused.
2,send the updated presentation before evening,Please send the updated presentation by the end of the day.,Could you please share the updated version of the presentation by tomorrow afternoon?,Could you please share the updated version of the presentation by tomorrow afternoon?
3,thanks for helping me with the assignment,"Thank you for using my services to assist with your assignment. I'm glad to have been able to help. If you need any further assistance in the future, please don't hesitate to reach out. Have a great day!",Thank you for your assistance in completing the assignment. I appreciate it.,Thank you for your assistance in completing my assignment. I appreciate it.
4,i have completed the changes you suggested,I have completed the changes as per your suggestions.,I have completed the changes requested by you. Please let me know if any further adjustments are required.,I have completed the changes requested by you. Please review them when convenient.
5,can you please tell me when the internship results will be announced,"I'm sorry, but I need more information to provide a specific answer. Could you please clarify which internship you're referring to? Additionally, if there's any particular date in mind for your application, that would also help. Without this context, it's difficult to give a precise timeline. Please let me know so we can proceed with the correct timing.",Could you please let us know when we can expect to hear about the outcome of our internship application?,Could you please let me know when the internship results will be announced?
6,the deadline is too short and we need more time,The deadline is extremely tight and we require additional time to complete the task. Please consider extending the deadline for our benefit.,The current deadline appears to be quite tight. We would appreciate additional time to complete the task. Please let us know if any extensions are possible.,The current deadline appears to be quite tight. We would appreciate additional time to complete the task. Please let us know if any extensions are possible.
7,i did not understand the feedback given during the review,I'm sorry you didn't fully understand the feedback provided during the review. Could you please clarify what specific part of the feedback was unclear to you so I can better assist you?,I may have missed understanding some of the feedback provided during the review. Could you please clarify what was meant?,I may have missed understanding some of the feedback provided during the review. Could you please clarify what was meant?
8,please check the attached file and let me know if anything is missing,Please review the attached document and let me know if any additional information is needed.,Please review the attached document and let me know if any additional information is required.,Please review the attached document and let me know if any additional information is required.
9,i am interested in applying for the position,I am interested in applying for the position.,I would like to express my intere

In [49]:
import pandas as pd
import json
import os

project_dir = "/content/Professional_Response_Generator"

# Load benchmark and audit
benchmark = pd.read_csv(
    f"{project_dir}/version1_vs_version2_benchmark.csv"
)
audit = pd.read_csv(
    f"{project_dir}/meaning_preservation_audit.csv"
)

# Validate all 15 prompts
assert len(benchmark) == 15
assert len(audit) == 15

# Calculate meaning-preservation counts
def preservation_counts(column):
    counts = audit[column].value_counts()
    return {
        "Yes": int(counts.get("Yes", 0)),
        "Partial": int(counts.get("Partial", 0)),
        "No": int(counts.get("No", 0))
    }

v1_preservation = preservation_counts("v1_meaning_preserved")
v2_preservation = preservation_counts("v2_meaning_preserved")

v1_additions = int(
    (audit["v1_invented_information"] == "Yes").sum()
)
v2_additions = int(
    (audit["v2_invented_information"] == "Yes").sum()
)

summary = {
    "evaluation_prompts": len(benchmark),
    "base_model": "Qwen2.5-1.5B-Instruct",
    "version_1_training_examples": 182,
    "version_2_training_examples": 231,
    "version_1_final_validation_loss": 0.427459,
    "version_2_final_validation_loss": 0.414156,
    "validation_loss_reduction_percent": round(
        (0.427459 - 0.414156) / 0.427459 * 100, 2
    ),
    "version_1_meaning_preservation": v1_preservation,
    "version_2_meaning_preservation": v2_preservation,
    "version_1_unsupported_additions_flagged": v1_additions,
    "version_2_unsupported_additions_flagged": v2_additions
}

# Save summary
summary_path = f"{project_dir}/final_evaluation_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

# Save combined evaluation files
combined = benchmark.merge(
    audit,
    on="id",
    how="left"
)

combined.to_csv(
    f"{project_dir}/final_three_model_evaluation.csv",
    index=False
)

print("FINAL EVALUATION SUMMARY")
print(json.dumps(summary, indent=4))

print("\nSaved files:")
print(summary_path)
print(f"{project_dir}/final_three_model_evaluation.csv")

FINAL EVALUATION SUMMARY
{
    "evaluation_prompts": 15,
    "base_model": "Qwen2.5-1.5B-Instruct",
    "version_1_training_examples": 182,
    "version_2_training_examples": 231,
    "version_1_final_validation_loss": 0.427459,
    "version_2_final_validation_loss": 0.414156,
    "validation_loss_reduction_percent": 3.11,
    "version_1_meaning_preservation": {
        "Yes": 8,
        "Partial": 6,
        "No": 1
    },
    "version_2_meaning_preservation": {
        "Yes": 8,
        "Partial": 6,
        "No": 1
    },
    "version_1_unsupported_additions_flagged": 8,
    "version_2_unsupported_additions_flagged": 7
}

Saved files:
/content/Professional_Response_Generator/final_evaluation_summary.json
/content/Professional_Response_Generator/final_three_model_evaluation.csv


In [50]:
git init

SyntaxError: invalid syntax (2830201818.py, line 1)